In [1]:
import gzip
import json
import time
import numpy as np
import math
import re
from implementation import SimpleTfidf, SimpleBPE
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
path = "D:\\Study\\Study_Class\\Semester_7\\NLP\\Lab\\lab01\\c4-train.00000-of-01024-30K.json.gz"

with gzip.open(path, "rt", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

print(len(data))
print(data[0])

30000
{'text': 'Beginners BBQ Class Taking Place in Missoula!\nDo you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.\nHe will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.\nThe cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.', 'timestamp': '2019-04-25T12:57:54Z', 'url': 'https://klyq.com/beginners-bbq-class-taking-place-in-missoula/'}


# Part D 

## 7.2

In [3]:
documents = [item["text"] for item in data]

print("number of docs: ", len(documents))
print("example the first doc: ", documents[0])

number of docs:  30000
example the first doc:  Beginners BBQ Class Taking Place in Missoula!
Do you want to get better at making delicious BBQ? You will have the opportunity, put this on your calendar now. Thursday, September 22nd join World Class BBQ Champion, Tony Balay from Lonestar Smoke Rangers. He will be teaching a beginner level class for everyone who wants to get better with their culinary skills.
He will teach you everything you need to know to compete in a KCBS BBQ competition, including techniques, recipes, timelines, meat selection and trimming, plus smoker and fire information.
The cost to be in the class is $35 per person, and for spectators it is free. Included in the cost will be either a t-shirt or apron and you will be tasting samples of each meat that is prepared.


In [4]:
tfidf = SimpleTfidf()

corpus_vocabulary = tfidf.build_vocabulary(documents)
N = len(documents)
V = len(corpus_vocabulary)

print(f"N (documents) = {N}")
print(f"V (vocabulary) = {V}")

N (documents) = 30000
V (vocabulary) = 193837


In [5]:
print(f"Number of documents = {N}")
print(f"Vocabulary size     = {V}")
print(f"Matrix shape         = ({N}, {V})")

Number of documents = 30000
Vocabulary size     = 193837
Matrix shape         = (30000, 193837)


## 7.4 

In [6]:
X_vectorizer = CountVectorizer(
    tokenizer=tfidf.tokenize,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

X_counts = X_vectorizer.fit_transform(documents).astype(float)
row_sums = np.asarray(X_counts.sum(axis=1)).ravel()
row_sums[row_sums == 0] = 1.0
X_tf = X_counts.multiply((1.0 / row_sums)[:, None]).tocsr()

X_df = np.asarray((X_counts > 0).sum(axis=0)).ravel()
X_idf = np.zeros(len(X_df), dtype=float)
valid_terms = X_df > 0
X_idf[valid_terms] = np.log(N / X_df[valid_terms])

X_tfidf = X_tf.multiply(X_idf.reshape(1, -1)).tocsr()
X_tfidf.eliminate_zeros()

nnz = X_tfidf.nnz
S = 1 - nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])
print(f"nnz(X) = {nnz}")
print(f"N x V = {N * V}")
print(f"Sparsity S = {S:.6f}")

nnz(X) = 5104560
N x V = 5815110000
Sparsity S = 0.999122


### Giải thích

Một document chỉ chứa một phần nhỏ các term trong vocabulary, nhưng vector vẫn có chiều `V` vì tất cả documents phải được biểu diễn trong cùng một không gian đặc trưng.

Mỗi chiều tương ứng với một term trong toàn bộ vocabulary. Nếu một term không xuất hiện trong document, giá trị tại chiều đó bằng 0.

Do đó, vector có nhiều phần tử bằng 0 và được gọi là sparse vector. Cách biểu diễn này giúp các document có thể được so sánh bằng cosine similarity.

## 7.5

In [7]:
corpus_vocabulary = tfidf.build_vocabulary(documents)

N = len(documents)
V = len(corpus_vocabulary)

corpus_df_counts = tfidf.compute_df(
    documents,
    corpus_vocabulary
)

corpus_idf = tfidf.compute_idf(
    corpus_df_counts,
    N
)

top_df_idx = np.argsort(corpus_df_counts)[::-1][:20]

print("The 20 most common terms by DF")

for index in top_df_idx:
    print(
        f"{corpus_vocabulary[index]:20s} "
        f"df={int(corpus_df_counts[index])}"
    )

top_idf_idx = np.argsort(corpus_idf)[::-1][:20]

print("\nThe 20 terms with the highest IDF")

for index in top_idf_idx:
    print(
        f"{corpus_vocabulary[index]:20s} "
        f"idf={corpus_idf[index]:.4f}"
    )

The 20 most common terms by DF
the                  df=27893
and                  df=27423
to                   df=26689
of                   df=26031
a                    df=25905
in                   df=25224
for                  df=23651
is                   df=22739
with                 df=21405
on                   df=20262
that                 df=18370
this                 df=17840
are                  df=17594
it                   df=17168
s                    df=16959
as                   df=16467
at                   df=16347
from                 df=16316
be                   df=16153
you                  df=16094

The 20 terms with the highest IDF
𐌼𐌿𐌽𐌳𐍃                idf=10.3090
hidi                 idf=10.3090
hingus               idf=10.3090
hingucker            idf=10.3090
hinesc               idf=10.3090
hindware             idf=10.3090
hindutva             idf=10.3090
hindustani           idf=10.3090
hindrichs            idf=10.3090
hindraf              idf=10.3090
hind

In [8]:
doc_index = 0
chosen_doc = documents[doc_index]

counts_doc = tfidf.compute_counts(
    chosen_doc,
    corpus_vocabulary
)

tf_doc = tfidf.compute_tf(
    counts_doc
)

tfidf_doc = tfidf.compute_tfidf(
    tf_doc,
    corpus_idf
)

top_tfidf_idx = np.argsort(
    tfidf_doc
)[::-1][:20]

print(
    f"\nThe 20 terms with the highest "
    f"TF-IDF in document {doc_index}"
)

for index in top_tfidf_idx:
    print(
        f"{corpus_vocabulary[index]:20s} "
        f"tfidf={tfidf_doc[index]:.4f}"
    )


The 20 terms with the highest TF-IDF in document 0
bbq                  tfidf=0.1784
class                tfidf=0.0941
balay                tfidf=0.0787
kcbs                 tfidf=0.0734
meat                 tfidf=0.0725
lonestar             tfidf=0.0703
missoula             tfidf=0.0611
apron                tfidf=0.0566
smoker               tfidf=0.0562
timelines            tfidf=0.0525
trimming             tfidf=0.0516
spectators           tfidf=0.0505
rangers              tfidf=0.0490
22nd                 tfidf=0.0465
beginner             tfidf=0.0454
beginners            tfidf=0.0451
culinary             tfidf=0.0449
tasting              tfidf=0.0427
cost                 tfidf=0.0425
tony                 tfidf=0.0417


### Phân tích TF-IDF

Một term xuất hiện rất nhiều trong corpus không nhất thiết có TF-IDF cao. Nếu term xuất hiện trong nhiều documents, giá trị IDF của nó sẽ thấp. Vì:

\[
TFIDF(t,d)=TF(t,d)\times IDF(t)
\]

nên TF-IDF có thể thấp dù term xuất hiện nhiều lần trong một document.

Ngược lại, một term có IDF cao cũng không nhất thiết có TF-IDF cao trong mọi document. IDF cao chỉ cho biết term đó hiếm trong toàn corpus. Nếu term không xuất hiện hoặc xuất hiện rất ít trong một document thì TF của nó thấp, dẫn đến TF-IDF thấp hoặc bằng 0.

# Part E

In [9]:
test_documents = [
    "cat eats fish",
    "dog eats fish",
    "cat likes fish"
]

tfidf = SimpleTfidf()

In [10]:
test_vocabulary = tfidf.build_vocabulary(test_documents)

expected_vocabulary = ["cat", "dog", "eats", "fish", "likes"]

assert test_vocabulary == expected_vocabulary


counts = tfidf.compute_counts(test_documents[0], test_vocabulary)

expected_counts = np.array([1, 0, 1, 1, 0], dtype=float)

np.testing.assert_array_equal(counts, expected_counts)


tf = tfidf.compute_tf(counts)

expected_tf = np.array([1 / 3, 0, 1 / 3, 1 / 3, 0], dtype=float)

np.testing.assert_allclose(tf, expected_tf, atol=1e-9)

assert abs(tf[0] - 1 / 3) < 1e-9

df = tfidf.compute_df(test_documents, test_vocabulary)

expected_df = np.array([2, 1, 2, 3, 1], dtype=float)

np.testing.assert_array_equal(df, expected_df)


idf = tfidf.compute_idf(df, len(test_documents))

expected_idf = np.array([
    math.log(3 / 2),  
    math.log(3 / 1),  
    math.log(3 / 2), 
    0.0,              
    math.log(3 / 1)  
])

np.testing.assert_allclose(idf, expected_idf, atol=1e-9)


tfidf_vector = tfidf.compute_tfidf(tf, idf)

expected_tfidf = (expected_tf * expected_idf)

np.testing.assert_allclose(tfidf_vector, expected_tfidf, atol=1e-9)

similarity = tfidf.cosine_similarity([1, 1, 1], [1, 1, 0])
expected_similarity = 2 / math.sqrt(6)
assert abs(similarity - expected_similarity) < 1e-9


zero_similarity = tfidf.cosine_similarity([0, 0, 0], [1, 2, 3])
assert zero_similarity == 0.0


print("All Part E unit tests passed!")

All Part E unit tests passed!


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np



student_counts = np.array([
    tfidf.compute_counts(
        document,
        test_vocabulary
    )
    for document in test_documents
])

student_tf = np.array([
    tfidf.compute_tf(counts)
    for counts in student_counts
])

student_df = tfidf.compute_df(
    test_documents,
    test_vocabulary
)

student_idf = tfidf.compute_idf(
    student_df,
    len(test_documents)
)

student_tfidf = np.array([
    tfidf.compute_tfidf(
        tf_vector,
        student_idf
    )
    for tf_vector in student_tf
])



reference_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    lowercase=True,
    norm=None,
    smooth_idf=False
)

sklearn_tfidf_raw = (
    reference_vectorizer
    .fit_transform(test_documents)
    .toarray()
)

sklearn_vocabulary = (
    reference_vectorizer
    .get_feature_names_out()
    .tolist()
)


sklearn_idf_plus_one = (
    reference_vectorizer.idf_
)

sklearn_counts = (
    sklearn_tfidf_raw
    / sklearn_idf_plus_one
)

row_sums = sklearn_counts.sum(
    axis=1,
    keepdims=True
)

row_sums[row_sums == 0] = 1.0

sklearn_tf = (
    sklearn_counts / row_sums
)

sklearn_df = (
    sklearn_counts > 0
).sum(axis=0)

sklearn_idf = (
    sklearn_idf_plus_one - 1.0
)

sklearn_tfidf_aligned = (
    sklearn_tf * sklearn_idf
)


np.set_printoptions(
    precision=6,
    suppress=True
)

print("Vocabulary:")
print(test_vocabulary)

print("\nStudent Counts:")
print(student_counts)

print("\nSklearn Counts:")
print(sklearn_counts)

print("\nStudent TF:")
print(student_tf)

print("\nSklearn TF:")
print(sklearn_tf)

print("\nStudent DF:")
print(student_df)

print("\nSklearn DF:")
print(sklearn_df)

print("\nStudent IDF:")
print(student_idf)

print("\nSklearn IDF after removing +1:")
print(sklearn_idf)

print("\nStudent TF-IDF:")
print(student_tfidf)

print("\nSklearn TF-IDF after aligning conventions:")
print(sklearn_tfidf_aligned)

print("\nVocabulary matches:")
print(test_vocabulary == sklearn_vocabulary)

print("\nCounts match:")
print(
    np.allclose(
        student_counts,
        sklearn_counts,
        atol=1e-6
    )
)

print("\nTF matches:")
print(
    np.allclose(
        student_tf,
        sklearn_tf,
        atol=1e-6
    )
)

print("\nDF matches:")
print(
    np.allclose(
        student_df,
        sklearn_df,
        atol=1e-6
    )
)

print("\nIDF matches:")
print(
    np.allclose(
        student_idf,
        sklearn_idf,
        atol=1e-6
    )
)

print("\nTF-IDF matches:")
print(
    np.allclose(
        student_tfidf,
        sklearn_tfidf_aligned,
        atol=1e-6
    )
)

Vocabulary:
['cat', 'dog', 'eats', 'fish', 'likes']

Student Counts:
[[1. 0. 1. 1. 0.]
 [0. 1. 1. 1. 0.]
 [1. 0. 0. 1. 1.]]

Sklearn Counts:
[[1. 0. 1. 1. 0.]
 [0. 1. 1. 1. 0.]
 [1. 0. 0. 1. 1.]]

Student TF:
[[0.333333 0.       0.333333 0.333333 0.      ]
 [0.       0.333333 0.333333 0.333333 0.      ]
 [0.333333 0.       0.       0.333333 0.333333]]

Sklearn TF:
[[0.333333 0.       0.333333 0.333333 0.      ]
 [0.       0.333333 0.333333 0.333333 0.      ]
 [0.333333 0.       0.       0.333333 0.333333]]

Student DF:
[2. 1. 2. 3. 1.]

Sklearn DF:
[2 1 2 3 1]

Student IDF:
[0.405465 1.098612 0.405465 0.       1.098612]

Sklearn IDF after removing +1:
[0.405465 1.098612 0.405465 0.       1.098612]

Student TF-IDF:
[[0.135155 0.       0.135155 0.       0.      ]
 [0.       0.366204 0.135155 0.       0.      ]
 [0.135155 0.       0.       0.       0.366204]]

Sklearn TF-IDF after aligning conventions:
[[0.135155 0.       0.135155 0.       0.      ]
 [0.       0.366204 0.135155 0.       0

### Comparison with Reference Implementation

Hai implementation cho kết quả tương đương khi sử dụng cùng công thức:

\[
TF(t,d)=
\frac{count(t,d)}
{\sum_{t'}count(t',d)}
\]

\[
IDF(t)=\log\left(\frac{N}{DF(t)}\right)
\]

\[
TFIDF(t,d)=TF(t,d)\times IDF(t)
\]

Nếu kết quả khác nhau, nguyên nhân có thể là:

- TF của thư viện sử dụng sublinear scaling:

\[
TF(t,d)=1+\log(count(t,d))
\]

- IDF của thư viện có smoothing:

\[
IDF(t)=
\log\left(\frac{1+N}{1+DF(t)}\right)+1
\]

- IDF không smoothing nhưng có cộng thêm \(1\):

\[
IDF(t)=\log\left(\frac{N}{DF(t)}\right)+1
\]

- Vector được L2-normalize:

\[
\hat{x}=\frac{x}{\|x\|_2}
\]

Ngoài ra, kết quả có thể khác do vocabulary ordering, tokenizer hoặc chi tiết
implementation.

# Part F

## 9.1. Pipeline A

In [12]:
def tokenize_pipeline_A(text):
    return text.lower().split()

def build_vocabulary_pipeline_A(documents):
    tokens = set()
    for doc in documents:
        tokens.update(tokenize_pipeline_A(doc))
    return sorted(tokens)

vocab_A = build_vocabulary_pipeline_A(documents)
print("Vocabulary size (Pipeline A): ", len(vocab_A))

Vocabulary size (Pipeline A):  473388


## 9.2. Pipeline B

In [13]:
def get_auto_stopwords(documents, vocabulary, df_counts, threshold_ratio=0.05):
    N_docs = len(documents)
    threshold = N_docs * threshold_ratio
    
    stopwords = set()
    for i, term in enumerate(vocabulary):
        if df_counts[i] > threshold:
            stopwords.add(term)
    return stopwords


auto_stopwords = get_auto_stopwords(
    documents,
    corpus_vocabulary,
    corpus_df_counts,
    threshold_ratio=0.05
)

print(f"Số lượng stopword tự động phát hiện: {len(auto_stopwords)}")
print("Danh sách:", sorted(auto_stopwords)[:30])

Số lượng stopword tự động phát hiện: 505
Danh sách: ['000', '1', '10', '100', '11', '12', '15', '2', '20', '25', '3', '30', '4', '5', '50', '6', '7', '8', '9', 'a', 'able', 'about', 'above', 'access', 'according', 'across', 'actually', 'add', 'added', 'addition']


In [14]:
def tokenize_pipeline_B(text, stopwords_set):
    text = text.lower()
    tokens = tfidf.tokenize(text)
    tokens = [t for t in tokens if t not in stopwords_set]
    return tokens

def build_vocabulary_pipeline_B(documents, stopwords_set):
    tokens = set()
    for doc in documents:
        tokens.update(tokenize_pipeline_B(doc, stopwords_set))
    return sorted(tokens)

vocab_B = build_vocabulary_pipeline_B(documents, auto_stopwords) 
print(f"Vocabulary size Pipeline B: {len(vocab_B)}")

Vocabulary size Pipeline B: 193332


## 9.3. Pipeline C

In [15]:
bpe = SimpleBPE(num_merges=100, min_frequency=2, lowercase=True)
bpe.fit(documents)

def tokenize_pipeline_C(text):
    return bpe.tokenize_text(text)

tokenized_documents_C = [
    tokenize_pipeline_C(document)
    for document in documents
]

vocab_C = sorted({
    token
    for tokens in tokenized_documents_C
    for token in tokens
})

print(f"Vocabulary size Pipeline C: {len(vocab_C)}")

Vocabulary size Pipeline C: 2693


## 9.4. So sánh

In [16]:
if "tokenize_pipeline_C" not in globals():
    raise RuntimeError("Run the Pipeline C cell first.")

def build_tfidf_for_pipeline(tokenizer):
    vectorizer = CountVectorizer(
        tokenizer=tokenizer,
        preprocessor=None,
        token_pattern=None,
        lowercase=False
    )

    counts = vectorizer.fit_transform(documents).astype(float)

    row_sums = np.asarray(counts.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0
    tf = counts.multiply((1.0 / row_sums)[:, None]).tocsr()

    number_of_documents = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    idf = np.zeros(len(df), dtype=float)
    valid_terms = df > 0
    idf[valid_terms] = np.log(
        number_of_documents / df[valid_terms]
    )

    tfidf = tf.multiply(idf.reshape(1, -1)).tocsr()
    tfidf.eliminate_zeros()
    document_norms = np.sqrt(
        np.asarray(tfidf.multiply(tfidf).sum(axis=1)).ravel()
    )
    return vectorizer, idf, tfidf, document_norms


def compute_oov_rate(texts, tokenizer, vocabulary):
    vocabulary = set(vocabulary)
    total_tokens = 0
    unknown_tokens = 0

    for text in texts:
        tokens = tokenizer(text)
        total_tokens += len(tokens)
        unknown_tokens += sum(token not in vocabulary for token in tokens)

    return unknown_tokens / total_tokens if total_tokens else 0.0


def benchmark_pipeline_search(vectorizer, idf, tfidf_matrix, document_norms, queries):
    durations = []

    for query in queries:
        start = time.perf_counter()
        query_counts = vectorizer.transform([query]).astype(float)
        query_total = float(np.asarray(query_counts.sum()).ravel()[0])

        if query_total > 0:
            query_tf = query_counts.multiply(1.0 / query_total)
            query_vector = query_tf.multiply(idf.reshape(1, -1)).tocsr()
            query_norm = np.sqrt(
                float(query_vector.multiply(query_vector).sum())
            )

            if query_norm > 0:
                scores = tfidf_matrix @ query_vector.T
                scores = np.asarray(scores.toarray()).ravel()
                denominator = document_norms * query_norm
                scores = np.divide(
                    scores,
                    denominator,
                    out=np.zeros_like(scores),
                    where=denominator != 0
                )
                np.argsort(scores)[::-1][:5]

        durations.append((time.perf_counter() - start) * 1000)

    return float(np.mean(durations)) if durations else 0.0


tokenizer_A = tokenize_pipeline_A
tokenizer_B = lambda text: tokenize_pipeline_B(text, auto_stopwords)
tokenizer_C = tokenize_pipeline_C

pipeline_tokenizers = {
    "Pipeline A": tokenizer_A,
    "Pipeline B": tokenizer_B,
    "Pipeline C": tokenizer_C,
}

pipeline_data = {}
evaluation_queries = globals().get("queries", [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "computer vision medical imaging",
])

for name, tokenizer in pipeline_tokenizers.items():
    vectorizer, idf, tfidf_matrix, document_norms = build_tfidf_for_pipeline(tokenizer)
    vocabulary = vectorizer.get_feature_names_out()
    average_tokens = np.mean([
        len(tokenizer(document))
        for document in documents
    ])
    sparsity = 1 - (
        tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
    )
    oov_rate = compute_oov_rate(
        evaluation_queries,
        tokenizer,
        vocabulary
    )
    search_ms = benchmark_pipeline_search(
        vectorizer,
        idf,
        tfidf_matrix,
        document_norms,
        evaluation_queries
    )

    pipeline_data[name] = {
        "vectorizer": vectorizer,
        "idf": idf,
        "tfidf_matrix": tfidf_matrix,
        "document_norms": document_norms,
        "vocabulary": vocabulary,
        "vocabulary_size": len(vocabulary),
        "average_tokens": average_tokens,
        "sparsity": sparsity,
        "oov_rate": oov_rate,
        "search_ms": search_ms,
    }

print(f"{'Metric':<30}{'Pipeline A':>18}{'Pipeline B':>18}{'Pipeline C':>18}")
print("-" * 84)
for metric, key, formatter in [
    ("Vocabulary size", "vocabulary_size", lambda x: f"{x:d}"),
    ("Average tokens/document", "average_tokens", lambda x: f"{x:.2f}"),
    ("Matrix sparsity", "sparsity", lambda x: f"{x:.6f}"),
    ("OOV rate", "oov_rate", lambda x: f"{x:.6f}"),
    ("Search time (ms/query)", "search_ms", lambda x: f"{x:.3f}"),
]:
    values = [
        formatter(pipeline_data[name][key])
        for name in ["Pipeline A", "Pipeline B", "Pipeline C"]
    ]
    print(f"{metric:<30}{values[0]:>18}{values[1]:>18}{values[2]:>18}")


Metric                                Pipeline A        Pipeline B        Pipeline C
------------------------------------------------------------------------------------
Vocabulary size                           473388            193332              2693
Average tokens/document                   361.09            151.49           1260.74
Matrix sparsity                         0.999611          0.999484          0.959530
OOV rate                                0.000000          0.000000          0.000000
Search time (ms/query)                    20.818             9.698            24.738


## 9.5. Analysis

### 1. Lowercasing làm thay đổi vocabulary như thế nào?

Lowercasing giúp các biến thể khác nhau về chữ hoa và chữ thường được ánh xạ
vào cùng một term. Vì vậy, số lượng unique terms có xu hướng giảm.

Tuy nhiên, trong thí nghiệm này Pipeline A và Pipeline B không chỉ khác nhau ở
lowercasing mà còn khác nhau về tokenization, punctuation handling và stopword
removal. Do đó, không thể quy toàn bộ sự khác biệt vocabulary chỉ cho
lowercasing.

Kết quả cho thấy vocabulary của Pipeline A lớn hơn đáng kể Pipeline B:
khoảng \(473{,}000\) so với \(193{,}000\) terms.

### 2. Stopword removal có luôn cải thiện representation không?

Không. Stopword removal làm giảm vocabulary và số token trong mỗi document,
qua đó giảm chi phí lưu trữ và tính toán.

Tuy nhiên, một số stopword có thể mang thông tin ngữ pháp hoặc xuất hiện trong
query. Nếu loại bỏ chúng, hệ thống có thể làm giảm recall hoặc bỏ sót các
document liên quan.

Vì vậy, stopword removal chỉ được xem là cải thiện khi các metric search như
Precision@5, Recall@5 hoặc MRR cũng được cải thiện.

### 3. Việc loại punctuation có thể làm mất thông tin gì?

Punctuation có thể biểu diễn ranh giới từ, câu, số thập phân, từ viết tắt,
URL, mã nguồn hoặc các biểu thức chuyên ngành.

Do đó, loại bỏ punctuation có thể làm mất một phần thông tin ngữ cảnh hoặc
làm thay đổi cách phân tách token. Việc loại punctuation làm vocabulary giảm,
nhưng vocabulary nhỏ hơn không đồng nghĩa với representation tốt hơn.

### 4. Pipeline nào tạo ra sparse matrix nhất?

Pipeline A tạo ra sparse matrix nhất vì:

\[
S_A=0.999611
\]

cao hơn:

\[
S_B=0.999484,\qquad S_C=0.959530
\]

Pipeline A có vocabulary lớn nhất nhưng mỗi document chỉ sử dụng một phần rất
nhỏ vocabulary. Pipeline C có vocabulary nhỏ hơn nhưng sử dụng nhiều subword
token hơn trong mỗi document, nên tỷ lệ zero thấp hơn.

### 5. Pipeline nào cho search tốt nhất?

Không thể kết luận chỉ dựa trên vocabulary size hoặc matrix sparsity. Pipeline
cho search tốt nhất phải được xác định bằng các metric retrieval được đo trên
cùng một evaluation set, chẳng hạn:

- Precision@5;
- Recall@5;
- MRR.

Các kết quả hiện tại chủ yếu cho thấy sự khác biệt về representation. Cần so
sánh trực tiếp search performance của cả ba pipeline trước khi kết luận
pipeline nào tốt nhất.

### 6. Search tốt hơn có đồng nghĩa với vocabulary nhỏ hơn không?

Không. Vocabulary size chỉ phản ánh số lượng feature, trong khi search quality
phụ thuộc vào khả năng giữ lại thông tin quan trọng và mức độ phù hợp giữa
document với query.

Ví dụ, Pipeline C có vocabulary nhỏ nhất nhưng average tokens/document cao
nhất và sparsity thấp hơn Pipeline A. Vì vậy, vocabulary nhỏ hơn không tự động
dẫn đến search tốt hơn.

# Part G — Application: Build a Document Search Engine

In [17]:
def build_sparse_index(documents, tfidf):
    vocabulary = tfidf.build_vocabulary(documents)
    term_to_index = {term: index for index, term in enumerate(vocabulary)}

    df = tfidf.compute_df(documents, vocabulary)
    idf = tfidf.compute_idf(df, len(documents))
    document_vectors = []
    document_norms = []

    for document in documents:
        counts = tfidf.compute_sparse_counts(document, term_to_index)
        total_tokens = sum(counts.values())
        vector = {}
        if total_tokens > 0:
            for term_index, count in counts.items():
                tf = count / total_tokens
                value = tf * idf[term_index]

                if value != 0:
                    vector[term_index] = value

        document_vectors.append(vector)
        document_norms.append(
            float(np.sqrt(sum(value ** 2 for value in vector.values())))
        )

    return {
        "vocabulary": vocabulary,
        "term_to_index": term_to_index,
        "idf": idf,
        "document_vectors": document_vectors,
        "document_norms": document_norms
    }

In [18]:
def transform_query(query, index, tfidf):
    counts = tfidf.compute_sparse_counts(
        query,
        index["term_to_index"]
    )
    total_tokens = sum(counts.values())
    query_vector = {}

    if total_tokens == 0:
        return query_vector

    for term_index, count in counts.items():
        tf = count / total_tokens
        value = tf * index["idf"][term_index]

        if value != 0:
            query_vector[term_index] = value

    return query_vector

In [19]:
def search(query, index, tfidf, documents, top_k=5):
    query_vector = transform_query(query, index, tfidf)
    if not query_vector:
        return []

    query_norm = np.sqrt(sum(value ** 2 for value in query_vector.values()))

    scores = []

    for document_id, document_vector in enumerate(index["document_vectors"]):
        dot_product = sum(
            query_value
            * document_vector.get(term_index, 0.0)
            for term_index, query_value
            in query_vector.items()
        )

        document_norm = index["document_norms"][document_id]
        if document_norm == 0:
            score = 0.0
        else:
            score = dot_product / (query_norm * document_norm)
        scores.append((document_id, score))

    ranking = sorted(scores, key=lambda item: item[1], reverse=True)

    return [
        {
            "rank": rank,
            "document_id": document_id,
            "score": score,
            "preview": documents[document_id][:200]
        }
        for rank, (document_id, score)
        in enumerate(ranking[:top_k], start=1)
    ]

In [20]:
tfidf = SimpleTfidf()
index = build_sparse_index(documents, tfidf)

results = search(
    "medical image classification",
    index,
    tfidf,
    documents,
    top_k=5
)

for result in results:
    print(result)

{'rank': 1, 'document_id': 18971, 'score': 0.44409148636741114, 'preview': 'The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in an environmentally responsible manner. The enviro'}
{'rank': 2, 'document_id': 8527, 'score': 0.3671536563139361, 'preview': 'History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico.'}
{'rank': 3, 'document_id': 19908, 'score': 0.23365401718297651, 'preview': 'Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-screen and HD resolution.\nRight click on image, and select “Save Image as League Of Legends Wallpapers” t'}
{'rank': 4, 'document_id': 15682, 'score': 0.226000508753156, 'preview': "- Group Image: Provided functionality of group's image, user can now view and change the group's image also.\n- Profile image: User can now set a profile image du

In [21]:
results = search(
    "transformer language model",
    index,
    tfidf,
    documents,
    top_k=5
)

for result in results:
    print(result)

{'rank': 1, 'document_id': 27936, 'score': 0.5004874750526543, 'preview': 'hi, I am having problems with transformer / circuit board om my Hobby 720 uml model 2003.\nIn view of the problems, can I bypass the transformer/circuit board ??'}
{'rank': 2, 'document_id': 25428, 'score': 0.2674955133597558, 'preview': "Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language your phone is set up to use, so to change it you have to pick "}
{'rank': 3, 'document_id': 24482, 'score': 0.20581191266084797, 'preview': 'Harald, you are a co-owner of Language Partners. Are you a linguist yourself, and how did you get into this competitive line of business?\nNo, I’m not a linguist; I am a historian and a business man. I'}
{'rank': 4, 'document_id': 701, 'score': 0.20160158782585583, 'preview': 'Program in Teaching French as a Foreign Language was established in 1985 under the Department of Foreign Education. 2 full profes

In [22]:
results = search(
    "deep learning healthcare",
    index,
    tfidf,
    documents,
    top_k=5
)

for result in results:
    print(result)

{'rank': 1, 'document_id': 11119, 'score': 0.3077949954114487, 'preview': 'The opportunities offered by Big Data will only materialize when healthcare systems move beyond the mere collection of large amounts of data. Linkage of previously separated data sets and their analys'}
{'rank': 2, 'document_id': 11979, 'score': 0.307256192741958, 'preview': 'Doctorate of Healthcare Organization Program is designed for the healthcare professionals who are working in healthcare organizations especially in hospitals or in clinics and who would like to broade'}
{'rank': 3, 'document_id': 6123, 'score': 0.3059142475926899, 'preview': 'With today’s advancement in technology, it is becoming more convenient than ever before to earn a Bachelor’s degree in a health-related field online. Some specializations, such as nutrition or public '}
{'rank': 4, 'document_id': 9252, 'score': 0.29733922096138377, 'preview': 'SAN DIEGO AND WASHINGTON, D.C. – Sept. 5, 2018 – West Health, a family of nonpartisan, nonpro

In [23]:
results = search(
    "natural language processing",
    index,
    tfidf,
    documents,
    top_k=5
)

for result in results:
    print(result)

{'rank': 1, 'document_id': 25428, 'score': 0.37410930323452707, 'preview': "Note: If you're on an iPhone, you cannot change the language of Facebook through the mobile app. Instead, Facebook uses whatever language your phone is set up to use, so to change it you have to pick "}
{'rank': 2, 'document_id': 8705, 'score': 0.3690188772646564, 'preview': 'These regulations may be called the Food Safety and Standards (Food Products Standards and Food Additives) Amendment Regulations, 2019. They shall come into force on the date of their publication in t'}
{'rank': 3, 'document_id': 24482, 'score': 0.28784090722061095, 'preview': 'Harald, you are a co-owner of Language Partners. Are you a linguist yourself, and how did you get into this competitive line of business?\nNo, I’m not a linguist; I am a historian and a business man. I'}
{'rank': 4, 'document_id': 701, 'score': 0.2819525030727195, 'preview': 'Program in Teaching French as a Foreign Language was established in 1985 under the Departme

# Part H — Evaluation


In [24]:
queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "computer vision medical imaging"
]

In [25]:
def build_pseudo_evaluation_set(queries, documents, tfidf, min_shared_terms=2, min_overlap=0.5):
    evaluation_set = []
    for query in queries:
        query_terms = set(tfidf.tokenize(query))
        relevant_ids = set()
        for document_id, document in enumerate(documents):
            document_terms = set(tfidf.tokenize(document))
            shared_terms = (query_terms & document_terms)
            overlap = (len(shared_terms) / len(query_terms) if query_terms else 0.0)
            if (len(shared_terms) >= min_shared_terms and overlap >= min_overlap):
                relevant_ids.add(document_id)
        evaluation_set.append({"query": query, "relevant": relevant_ids})
    return evaluation_set

In [26]:
evaluation_set = (build_pseudo_evaluation_set(queries, documents, tfidf))
for item in evaluation_set:
    print(item["query"], "->", len(item["relevant"]), "relevant documents")

medical image classification -> 51 relevant documents
transformer language model -> 69 relevant documents
deep learning healthcare -> 145 relevant documents
natural language processing -> 154 relevant documents
computer vision medical imaging -> 158 relevant documents


In [27]:
def evaluate_sparse_index(index, tfidf, documents, evaluation_set, k=5):
    precision_values = []
    recall_values = []
    reciprocal_ranks = []

    for item in evaluation_set:
        query = item["query"]
        relevant_ids = set(item["relevant"])
        results = search(query, index, tfidf, documents, top_k=k)
        retrieved_ids = [result["document_id"] for result in results]
        hits = (set(retrieved_ids) & relevant_ids)
        precision_values.append(len(hits) / k)
        recall_values.append(len(hits) / len(relevant_ids) if relevant_ids else 0.0)

        reciprocal_rank = 0.0
        for rank, document_id in enumerate(retrieved_ids, start=1):
            if document_id in relevant_ids:
                reciprocal_rank = 1 / rank
                break
        reciprocal_ranks.append(reciprocal_rank)

    return {
        "P@5": sum(precision_values)
        / len(precision_values),

        "Recall@5": sum(recall_values)
        / len(recall_values),

        "MRR": sum(reciprocal_ranks)
        / len(reciprocal_ranks)
    }



In [28]:
metrics = evaluate_sparse_index(
    index,
    tfidf,
    documents,
    evaluation_set,
    k=5
)

print("Part H Evaluation")
print(f"P@5:      {metrics['P@5']:.4f}")
print(f"Recall@5: {metrics['Recall@5']:.4f}")
print(f"MRR:      {metrics['MRR']:.4f}")

Part H Evaluation
P@5:      0.2400
Recall@5: 0.0095
MRR:      0.4667


In [29]:
for item in evaluation_set:
    query = item["query"]
    relevant_ids = set(
        item["relevant"]
    )

    results = search(
        query,
        index,
        tfidf,
        documents,
        top_k=5
    )

    print("=" * 80)
    print("QUERY:", query)

    for result in results:
        document_id = result["document_id"]

        label = (
            "CORRECT"
            if document_id in relevant_ids
            else "ERROR"
        )

        print(
            f"[{label}] "
            f"Rank={result['rank']} "
            f"Doc={document_id} "
            f"Score={result['score']:.4f}"
        )

        print(result["preview"])

QUERY: medical image classification
[ERROR] Rank=1 Doc=18971 Score=0.4441
The new RTS Environmental Classification system (RTS GLT) is designed for parties who are commissioning construction projects and who want to build in an environmentally responsible manner. The enviro
[ERROR] Rank=2 Doc=8527 Score=0.3672
History of maize classification. How races used in classification. Geographical distribution. Existing races of maize in Mexico.
[ERROR] Rank=3 Doc=19908 Score=0.2337
Download League Of Legends Wallpapers in high-quality for your desktop and smart-phone in wide-screen and HD resolution.
Right click on image, and select “Save Image as League Of Legends Wallpapers” t
[ERROR] Rank=4 Doc=15682 Score=0.2260
- Group Image: Provided functionality of group's image, user can now view and change the group's image also.
- Profile image: User can now set a profile image during sign up process.
This is a great a
[ERROR] Rank=5 Doc=8370 Score=0.2206
What is a Online Medical Second Opinion?
For

In [30]:
import csv
import time
from pathlib import Path

required_variables = [
    "pipeline_data",
    "documents",
    "tfidf",
]

for variable in required_variables:
    if variable not in globals():
        raise RuntimeError(
            f"Thiếu biến {variable}. "
            "Hãy chạy các cell Pipeline A-C trước."
        )


queries_for_csv = globals().get("queries", [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing",
    "computer vision medical imaging",
])

if (
    "evaluation_set" not in globals()
    or len(evaluation_set) != len(queries_for_csv)
    or [
        item["query"]
        for item in evaluation_set
    ] != queries_for_csv
):
    evaluation_set_for_csv = build_pseudo_evaluation_set(
        queries_for_csv,
        documents,
        tfidf
    )
else:
    evaluation_set_for_csv = evaluation_set


def search_pipeline(
    query,
    pipeline_info,
    documents,
    top_k=5
):
    vectorizer = pipeline_info["vectorizer"]
    idf = pipeline_info["idf"]
    tfidf_matrix = pipeline_info["tfidf_matrix"]

    if "document_norms" in pipeline_info:
        document_norms = pipeline_info["document_norms"]
    else:
        document_norms = np.sqrt(
            np.asarray(
                tfidf_matrix.multiply(tfidf_matrix).sum(axis=1)
            ).ravel()
        )

    start_time = time.perf_counter()

    query_counts = vectorizer.transform([query]).astype(float)

    query_total = float(
        np.asarray(query_counts.sum()).ravel()[0]
    )

    if query_total == 0:
        return [], 0.0

    query_tf = query_counts.multiply(1.0 / query_total)

    query_vector = query_tf.multiply(
        idf.reshape(1, -1)
    ).tocsr()

    query_norm = np.sqrt(
        float(
            query_vector.multiply(query_vector).sum()
        )
    )

    if query_norm == 0:
        return [], 0.0

    scores = tfidf_matrix @ query_vector.T
    scores = np.asarray(scores.toarray()).ravel()

    denominator = document_norms * query_norm

    scores = np.divide(
        scores,
        denominator,
        out=np.zeros_like(scores),
        where=denominator != 0
    )

    document_ids = np.arange(len(scores))

    ranked_ids = np.lexsort(
        (document_ids, -scores)
    )[:top_k]

    elapsed_ms = (
        time.perf_counter() - start_time
    ) * 1000

    results = []

    for rank, document_id in enumerate(
        ranked_ids,
        start=1
    ):
        results.append({
            "rank": rank,
            "document_id": int(document_id),
            "similarity": float(scores[document_id]),
            "preview": documents[document_id][:200]
        })

    return results, elapsed_ms


csv_rows = []

pipeline_names = [
    "Pipeline A",
    "Pipeline B",
    "Pipeline C",
]

for pipeline_name in pipeline_names:
    pipeline_info = pipeline_data[pipeline_name]

    precision_values = []
    recall_values = []
    reciprocal_ranks = []

    for evaluation_item in evaluation_set_for_csv:
        query = evaluation_item["query"]
        relevant_ids = set(
            evaluation_item["relevant"]
        )

        results, elapsed_ms = search_pipeline(
            query,
            pipeline_info,
            documents,
            top_k=5
        )

        retrieved_ids = [
            result["document_id"]
            for result in results
        ]

        hits = (
            set(retrieved_ids)
            & relevant_ids
        )

        precision_at_5 = len(hits) / 5

        recall_at_5 = (
            len(hits) / len(relevant_ids)
            if relevant_ids
            else 0.0
        )

        reciprocal_rank = 0.0

        for rank, document_id in enumerate(
            retrieved_ids,
            start=1
        ):
            if document_id in relevant_ids:
                reciprocal_rank = 1 / rank
                break

        precision_values.append(precision_at_5)
        recall_values.append(recall_at_5)
        reciprocal_ranks.append(reciprocal_rank)

        for result in results:
            csv_rows.append({
                "record_type": "retrieval",
                "pipeline": pipeline_name,
                "query": query,
                "rank": result["rank"],
                "document_id": result["document_id"],
                "similarity": f"{result['similarity']:.6f}",
                "preview": result["preview"].replace(
                    "\n",
                    " "
                ),
                "relevant_count": len(relevant_ids),
                "hits_at_5": "",
                "precision_at_5": "",
                "recall_at_5": "",
                "reciprocal_rank": "",
                "search_time_ms": f"{elapsed_ms:.3f}"
            })

        csv_rows.append({
            "record_type": "evaluation",
            "pipeline": pipeline_name,
            "query": query,
            "rank": "",
            "document_id": "",
            "similarity": "",
            "preview": "",
            "relevant_count": len(relevant_ids),
            "hits_at_5": len(hits),
            "precision_at_5": f"{precision_at_5:.6f}",
            "recall_at_5": f"{recall_at_5:.6f}",
            "reciprocal_rank": f"{reciprocal_rank:.6f}",
            "search_time_ms": f"{elapsed_ms:.3f}"
        })

    csv_rows.append({
        "record_type": "summary",
        "pipeline": pipeline_name,
        "query": "ALL",
        "rank": "",
        "document_id": "",
        "similarity": "",
        "preview": "",
        "relevant_count": "",
        "hits_at_5": "",
        "precision_at_5": (
            f"{np.mean(precision_values):.6f}"
        ),
        "recall_at_5": (
            f"{np.mean(recall_values):.6f}"
        ),
        "reciprocal_rank": (
            f"{np.mean(reciprocal_ranks):.6f}"
        ),
        "search_time_ms": ""
    })


results_path = Path(
    r"D:\Study\Study_Class\Semester_7\NLP\Lab\lab01\results.csv"
)

fieldnames = [
    "record_type",
    "pipeline",
    "query",
    "rank",
    "document_id",
    "similarity",
    "preview",
    "relevant_count",
    "hits_at_5",
    "precision_at_5",
    "recall_at_5",
    "reciprocal_rank",
    "search_time_ms",
]

with results_path.open(
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames
    )
    writer.writeheader()
    writer.writerows(csv_rows)


print(f"Đã lưu kết quả tại: {results_path}")
print(f"Tổng số dòng dữ liệu: {len(csv_rows)}")

Đã lưu kết quả tại: D:\Study\Study_Class\Semester_7\NLP\Lab\lab01\results.csv
Tổng số dòng dữ liệu: 93


# Part I — Error Analysis

## 1. Query có kết quả tương đối tốt

### Query: `deep learning healthcare`

Các kết quả đầu tiên chủ yếu liên quan đến lĩnh vực healthcare:

| Rank | Document ID | Similarity | Nhận xét |
|---:|---:|---:|---|
| 1 | 11119 | 0.3078 | Nội dung về healthcare systems và Big Data |
| 2 | 11979 | 0.3073 | Nội dung về healthcare organizations |
| 3 | 6123 | 0.3059 | Nội dung về healthcare professionals |
| 4 | 9252 | 0.2973 | Nội dung về healthcare research |
| 5 | 11777 | 0.2471 | Không liên quan rõ ràng đến healthcare |

Bốn document đầu có lexical overlap mạnh với term `healthcare`. Điều này cho
thấy TF-IDF có thể truy hồi tương đối tốt khi query và document sử dụng cùng
vocabulary. Tuy nhiên, term `healthcare` có thể đóng góp lớn hơn hai term
`deep` và `learning`, nên kết quả chưa chắc phản ánh đầy đủ ý nghĩa của toàn
bộ query.

Document ở rank 5 là một false positive, cho thấy lexical matching vẫn có thể
trả về document không liên quan.

### Query: `natural language processing`

Các kết quả trả về phần lớn có liên quan đến chủ đề language:

- Document 25428 chứa term `language`;
- Document 24482 nói về Language Partners;
- Document 701 nói về French language;
- Document 4075 nói về Spanish language.

Đây là kết quả tương đối tốt ở mức chủ đề, nhưng chưa đạt relevance hoàn toàn
về mặt ngữ nghĩa. Hệ thống nhận diện được term `language`, nhưng chưa phân biệt
được ngữ cảnh cụ thể của `natural language processing`. Document 8705 về food
safety là một false positive.

## 2. Query có kết quả kém

### Query: `transformer language model`

Document đứng đầu là:

    Document ID: 27936
    Similarity: 0.5005

Document này nói về transformer trong mạch điện và circuit board, không phải
transformer trong xử lý ngôn ngữ tự nhiên.

Nguyên nhân là term `transformer` có lexical overlap trực tiếp với query và
đóng góp lớn vào cosine similarity. Mô hình TF-IDF không hiểu được sự khác
nhau giữa electrical transformer và language model transformer.

Các document ở các rank tiếp theo cũng chủ yếu chứa term `language` hoặc
`transformer`, nhưng không nhất thiết nói về mô hình ngôn ngữ.

### Query: `medical image classification`

Document đứng đầu là:

    Document ID: 18971
    Similarity: 0.4441

Tuy nhiên, nội dung nói về một hệ thống classification trong xây dựng, không
phải medical image classification. Document ở rank 2 nói về classification
trong lịch sử ngô.

Các document này được xếp hạng cao vì chứa các term `classification` hoặc
`image`, nhưng không thể hiện đúng quan hệ ngữ nghĩa giữa `medical`, `image`
và `classification`.

## 3. Failure case quan trọng nhất

Failure case quan trọng nhất là query `transformer language model`. Đây là
minh chứng rõ ràng cho giới hạn của lexical representation: hai document có
thể cùng chứa một term nhưng thuộc các lĩnh vực hoàn toàn khác nhau.

Nguyên nhân chính của failure là lexical matching, không phải lỗi trong công
thức TF, IDF hoặc cosine similarity. Kết quả này cho thấy cần sử dụng
representation mang tính ngữ nghĩa hơn như word embedding, contextual
embedding hoặc Transformer-based representation.

Các relevance labels hiện tại được tạo bằng lexical overlap, vì vậy P@5,
Recall@5 và MRR cần được diễn giải thận trọng.